# 02. Logistic Regression - Titanic Survival Prediction## What is this notebook about?We predict whether a Titanic passenger **survived** or **died**, based on their age, ticket class, sex, and other info.Despite the name, **Logistic Regression** is a **classification** algorithm (yes/no), not regression. It uses a sigmoid (S-curve) to convert a linear combination of features into a probability between 0 and 1.## What you will learn1. How to handle a real, **messy** dataset (with missing values)2. **Feature engineering** to create more useful columns3. **One-hot encoding** for categorical features4. How to train Logistic Regression with a **Pipeline** (prevents data leakage)5. **Classification metrics**: accuracy, precision, recall, F1, ROC-AUC6. How to read a **confusion matrix** and **ROC curve**## DatasetThe classic Titanic dataset, built into seaborn. Columns include `survived` (target), `pclass` (ticket class), `sex`, `age`, `sibsp` (siblings/spouses aboard), `parch` (parents/children aboard), `fare`, `embarked` (port).Let's go!

In [ ]:
# If you're running locally, uncomment and run this once.# In Google Colab, all of these are pre-installed - you can skip this cell.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebookimport numpy as np                 # numerical arraysimport pandas as pd                # tabular data (DataFrames)import matplotlib.pyplot as plt    # plottingimport seaborn as sns              # prettier statistical plots# Set up plot stylesns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)# Set a random seed so results are reproduciblenp.random.seed(42)

## Step 1. Load and peek at the data

In [ ]:
# Load the Titanic datasetdf = sns.load_dataset("titanic")print(f"Shape: {df.shape}")df.head()

In [ ]:
# Check for missing values - this is critical before training!# isnull() returns True/False, sum() counts the Trues per columnprint("Missing values per column:")print(df.isnull().sum().sort_values(ascending=False).head(8))

**What we see:** `deck` is mostly missing (drop it). `age` has many missing values (we'll fill them in). `embarked` has just 2 missing values (fill with the most common port).## Step 2. Clean and engineer featuresWe will:1. Pick only the useful columns2. Create new features (`family_size`, `is_alone`)3. Fill missing values4. Convert categorical columns to numbers

In [ ]:
# Keep only the columns we plan to usedf = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()# Engineer new features from existing ones# family_size = number of family members onboard, including selfdf["family_size"] = df["sibsp"] + df["parch"] + 1# is_alone = 1 if travelling alone, else 0df["is_alone"] = (df["family_size"] == 1).astype(int)# Fill missing valuesdf["age"].fillna(df["age"].median(), inplace=True)              # use median agedf["embarked"].fillna(df["embarked"].mode()[0], inplace=True)   # use most common port# One-hot encode the categorical columns: sex and embarked# drop_first=True drops one dummy to avoid the "dummy variable trap"df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True)print(f"Shape after cleaning: {df.shape}")df.head()

**What changed:** `sex` became `sex_male` (1 if male, 0 if female). `embarked` became 2 columns: `embarked_Q` and `embarked_S`.## Step 3. Split into Training and Test setsWe hold out 20% of the data as a final test set the model will not see during training.

In [ ]:
from sklearn.model_selection import train_test_split# Separate features (X) from the target (y)X = df.drop(columns=["survived"])y = df["survived"]# Split. stratify=y keeps class proportions the same in train and test (important for imbalanced data)X_train, X_test, y_train, y_test = train_test_split(    X, y,    test_size=0.2,         # 20% for testing    random_state=42,       # reproducibility    stratify=y,            # preserve class ratio)print(f"Training set: {X_train.shape}")print(f"Test set:     {X_test.shape}")

## Step 4. Train Logistic Regression in a PipelineA **Pipeline** chains preprocessing and modelling into one object. Why?- It prevents **data leakage** (e.g. fitting the scaler on test data by accident).- It makes your code cleaner.- It treats the whole thing as one model for cross-validation.

In [ ]:
from sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.linear_model import LogisticRegression# Pipeline: first scale features (mean 0, std 1), then fit logistic regressionpipe = Pipeline([    ("scaler", StandardScaler()),                  # step 1: scale    ("model",  LogisticRegression(max_iter=1000)), # step 2: classify])# Train on the training data onlypipe.fit(X_train, y_train)# Make predictions on the test datay_pred  = pipe.predict(X_test)                  # 0 or 1 predictionsy_proba = pipe.predict_proba(X_test)[:, 1]      # probabilities of survival

## Step 5. EvaluateWe compute several metrics. Each tells us something different about the model.

In [ ]:
from sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    roc_auc_score, confusion_matrix, classification_report,    roc_curve,)print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}    (overall fraction correct)")print(f"Precision: {precision_score(y_test, y_pred):.3f}    (when we say survived, how often right)")print(f"Recall:    {recall_score(y_test, y_pred):.3f}    (of actual survivors, how many we caught)")print(f"F1 Score:  {f1_score(y_test, y_pred):.3f}    (harmonic mean of precision and recall)")print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}    (overall ranking quality)")print()print("Detailed report:")print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

## Step 6. Confusion MatrixA 2x2 grid showing how often we got each combination right or wrong.

In [ ]:
# Build and plot confusion matrixcm = confusion_matrix(y_test, y_pred)sns.heatmap(cm, annot=True, fmt="d", cmap="Greys",            xticklabels=["Died", "Survived"],            yticklabels=["Died", "Survived"])plt.xlabel("Predicted")plt.ylabel("Actual")plt.title("Confusion Matrix")plt.show()print("\nReading the matrix:")print(f"  True Negatives  (correctly predicted Died):     {cm[0, 0]}")print(f"  False Positives (predicted Survived, but Died): {cm[0, 1]}")print(f"  False Negatives (predicted Died, but Survived): {cm[1, 0]}")print(f"  True Positives  (correctly predicted Survived): {cm[1, 1]}")

## Step 7. ROC CurveThe ROC curve shows the trade-off between True Positive Rate and False Positive Rate as we vary the decision threshold. The **AUC** (area under the curve) summarizes this in one number from 0.5 (random) to 1.0 (perfect).

In [ ]:
# Compute ROC curve pointsfpr, tpr, thresholds = roc_curve(y_test, y_proba)plt.figure(figsize=(8, 6))plt.plot(fpr, tpr, color="black", lw=2, label=f"AUC = {roc_auc_score(y_test, y_proba):.2f}")plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random guessing")plt.xlabel("False Positive Rate")plt.ylabel("True Positive Rate")plt.title("ROC Curve")plt.legend()plt.show()

## Step 8. Inspect the CoefficientsLogistic regression is interpretable. Each coefficient tells us how a feature affects the predicted log-odds of survival.

In [ ]:
# Coefficients (positive = increases survival chance, negative = decreases)coef_df = pd.DataFrame({    "Feature": X.columns,    "Coefficient": pipe.named_steps["model"].coef_[0],}).sort_values("Coefficient", key=abs, ascending=False)print(coef_df)

**Typical findings:**- `sex_male` strongly **negative** (men less likely to survive - "women and children first")- `pclass` negative (higher class number = lower class = lower survival)- `fare` positive (more expensive ticket = better cabin = higher survival)These match what we know historically about the disaster.## Step 9. Exercises1. **Adjust the threshold** away from 0.5 (e.g. 0.3 or 0.7). How does it change precision and recall?   ```python   y_pred_threshold = (y_proba > 0.3).astype(int)   ```2. **Add L1 regularization** (Lasso) to do automatic feature selection:   ```python   LogisticRegression(penalty="l1", solver="liblinear", C=0.5)   ```3. **Try `class_weight="balanced"`** to handle the slight class imbalance.4. **Plot the calibration curve** (`sklearn.calibration.calibration_curve`) - are the predicted probabilities trustworthy?5. **Extract the `Title`** (Mr, Mrs, Miss, Master, Dr) from the original Kaggle Titanic data. It's often a strong predictor.When done, head to **03 - Regularization**!